# Annual climate stripes with CDO and NCO
### Yiwen Li · Fresh Eyes on CMIP

Start from **ACCESS-CM2 historical + SSP5-8.5** monthly temperature and calculate
the **annual Arctic temperature anomaly for 1979–2024**, relative to **1981–2010**.
Each processing command is followed by a short query that shows its result.

This lesson compares the final annual series and stripes. The
[complete workflow](00_yl_20_minute_binder_demo.ipynb) and
[hands-on notebook](04_monthly_anomaly_hands_on.ipynb) also calculate the twelve
monthly reference fields, monthly anomalies and the duration-weighted map.
The shared definitions follow
[Evgenia's recipe](https://github.com/egalytska/Fresh_eyes/blob/040b9c7962355b1f8fa1312b7af867b458102b85/ESMValTool/recipe_temperature_anomalies.yml).

## 01 · Data and calculation

| Item | Choice |
|---|---|
| Input | ACCESS-CM2 monthly near-surface temperature (`tas`, Kelvin) |
| Analysis | 1979–2024: 552 months, 46 complete years |
| Region | Latitude centres 65–90°N, all longitudes |
| Spatial mean | Equal weight for every selected grid point |
| Annual mean | Equal weight for each of the twelve monthly samples |
| Reference | Mean of the 30 annual temperatures in 1981–2010 |

**Annual anomaly = annual temperature − reference temperature.**

With complete calendar years and these fixed weights, this gives the same
annual result as calculating twelve monthly reference fields, subtracting the
matching reference from every month, and then averaging each year. It does not
produce the individual monthly anomaly fields or the weighted map.

Run the next cells with **Shift+Enter**, including collapsed cells. The data
cell creates a fresh results folder with `data/`, `cdo/` and `nco/` directories.
The same command text can be used in a terminal in that results folder.

In [ ]:
# Select the data and create this run
from pathlib import Path
import os, sys, subprocess, json, uuid, shlex
from datetime import datetime, timezone
from IPython.display import display, Image
from IPython.core.magic import no_var_expand

ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "scripts/run_python_updated.py").is_file()), None)
assert ROOT is not None, "Open this notebook from the workshop repository."
os.chdir(ROOT)

# The seminar uses real ACCESS-CM2. Synthetic mode is an explicit software-test option.
DATA_MODE = os.environ.get("WORKSHOP_MODE", "real")  # "real" or "synthetic"
END_YEAR = 2024
if os.environ.get("WORKSHOP_END_YEAR", "2024") != "2024":
    raise ValueError("This command-by-command lesson uses explicit 1979–2024 dates. Use Run_Updated_Demo.command --end-year 2100 for the extended batch calculation.")
assert DATA_MODE in {"synthetic", "real"}
assert END_YEAR >= 2024
os.environ["OMP_NUM_THREADS"] = "1"
if DATA_MODE == "synthetic":
    fixture = ROOT / "data/updated_fixture"
    subprocess.run([sys.executable, str(ROOT / "scripts/generate_updated_fixture.py"), str(fixture)], check=True)
    HIST, FUTURE = fixture / "historical.nc", fixture / "ssp585.nc"
else:
    directory = ROOT / "data/access_cm2"
    HIST = Path(os.environ.get("WORKSHOP_HISTORICAL", str(directory / "tas_Amon_ACCESS-CM2_historical_r1i1p1f1_gn_185001-201412.nc")))
    FUTURE = Path(os.environ.get("WORKSHOP_FUTURE", str(directory / "tas_Amon_ACCESS-CM2_ssp585_r1i1p1f1_gn_201501-210012.nc")))
    missing = [str(p) for p in (HIST, FUTURE) if not p.is_file()]
    if missing:
        raise FileNotFoundError("Real-data mode requires both raw files. No synthetic substitution is allowed. Missing: " + "; ".join(missing) + ". See docs/UPDATED_DEMO.md.")

HIST, FUTURE = HIST.resolve(), FUTURE.resolve()
RUN = ROOT / "outputs" / ("short_annual_" + DATA_MODE + "_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S") + "_" + uuid.uuid4().hex[:6])
RUN.mkdir(parents=True)
for directory in ("data", "cdo", "nco", "exercises"):
    (RUN / directory).mkdir()
(RUN / "data/historical.nc").symlink_to(HIST.resolve())
(RUN / "data/ssp585.nc").symlink_to(FUTURE.resolve())
os.chdir(RUN)

# Execute each visible command directly; no shell expansion or script wrapper.
# A failed command raises an exception, so Run All cannot silently continue.
def run_command(command):
    result = subprocess.run(shlex.split(command), stdin=subprocess.DEVNULL,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(result.stdout, end="")
    result.check_returncode()
ip = get_ipython()
ip.automagic = True
def register_tool(tool):
    # Preserve NCO array literals and expressions exactly as written.
    @no_var_expand
    def execute(arguments):
        run_command(tool + " " + arguments)
    ip.register_magic_function(execute, magic_kind="line", magic_name=tool)
for tool in ("cdo", "ncap2", "ncatted", "ncecat", "ncks", "ncra", "ncrcat", "ncrename", "ncwa", "ncdiff"):
    register_tool(tool)
print("DATA MODE:", "SYNTHETIC SOFTWARE TEST — NOT ACCESS-CM2" if DATA_MODE == "synthetic" else "REAL ACCESS-CM2: historical + SSP5-8.5")
print("Analysis:", f"1979–{END_YEAR}", "| Monthly reference: 1981–2010")
print("Outputs:", RUN.relative_to(ROOT))

In [ ]:
# Check data coverage and tool versions
_ = subprocess.run([sys.executable, str(ROOT / "scripts/check_updated_inputs.py"), str(HIST), str(FUTURE), "--end-year", str(END_YEAR)], check=True)
for command in (["cdo", "-V"], ["ncks", "--version"]):
    result = subprocess.run(command, check=True, text=True, capture_output=True)
    print((result.stdout + result.stderr).splitlines()[0])
print("Python", sys.version.split()[0])

## 02 · Independent reference

The reference calculation uses the complete monthly-anomaly method. We will
check that the shorter annual calculation gives the same 46 annual values.

In [ ]:
# Calculate the independent reference
_ = subprocess.run([sys.executable, str(ROOT / "scripts/run_python_updated.py"), str(HIST), str(FUTURE), str(RUN / "python"), "--end-year", str(END_YEAR)], check=True, capture_output=True, text=True)
print(f"Reference ready: five products for 1979–{END_YEAR}.")

## 03 · CDO

Read each command as an operation on an input file that writes an output file.
Run the next query to inspect the result before continuing.

### 03.01 · Select historical months

Keep 1979–2014: 432 monthly temperature fields.

In [ ]:
cdo selyear,1979/2014 data/historical.nc cdo/historical.nc

**Inspect the result**

In [ ]:
cdo ntime cdo/historical.nc

### 03.02 · Select future months

Keep 2015–2024 from SSP5-8.5: 120 monthly fields.

In [ ]:
cdo selyear,2015/2024 data/ssp585.nc cdo/ssp585.nc

**Inspect the result**

In [ ]:
cdo ntime cdo/ssp585.nc

### 03.03 · Join the experiments

Join the files in time order. -b F64 retains double precision for the following calculations.

In [ ]:
cdo -b F64 mergetime cdo/historical.nc cdo/ssp585.nc cdo/merged.nc

**Inspect the result**

In [ ]:
cdo ntime cdo/merged.nc

### 03.04 · Convert Kelvin values to Celsius

Subtract 273.15. The next step updates the unit label.

In [ ]:
cdo subc,273.15 cdo/merged.nc cdo/celsius_raw.nc

**Inspect the result**

In [ ]:
cdo infon -seltimestep,1 cdo/celsius_raw.nc

### 03.05 · Label the converted values

Set the unit to degC. The previous command changed the numbers; this command changes their label.

In [ ]:
cdo setunit,degC cdo/celsius_raw.nc cdo/celsius.nc

**Inspect the result**

In [ ]:
cdo showunit cdo/celsius.nc

### 03.06 · Select the Arctic

Select all longitudes and latitude centres from 65°N to 90°N.

In [ ]:
cdo sellonlatbox,0,360,65,90 cdo/celsius.nc cdo/arctic.nc

**Inspect the result**

In [ ]:
cdo ngridpoints cdo/arctic.nc

### 03.07 · Average the Arctic grid points

Give each selected grid point equal weight. The result contains one temperature per month.

In [ ]:
cdo fldmean,weights=false cdo/arctic.nc cdo/arctic_monthly.nc

**Inspect the result**

In [ ]:
cdo outputtab,date,value -seltimestep,1/3 cdo/arctic_monthly.nc

### 03.08 · Average each complete year

Give the twelve monthly samples equal weight. The result contains 46 annual temperatures.

In [ ]:
cdo yearmean cdo/arctic_monthly.nc cdo/annual_temperature.nc

**Inspect the result**

In [ ]:
cdo ntime cdo/annual_temperature.nc

### 03.09 · Select the reference years

The reference contains 30 annual temperatures from 1981 through 2010.

In [ ]:
cdo selyear,1981/2010 cdo/annual_temperature.nc cdo/baseline.nc

**Inspect the result**

In [ ]:
cdo ntime cdo/baseline.nc

### 03.10 · Calculate the reference temperature

Average those 30 years to obtain one reference temperature.

In [ ]:
cdo timmean cdo/baseline.nc cdo/reference.nc

**Inspect the result**

In [ ]:
cdo output cdo/reference.nc

### 03.11 · Subtract the reference

Annual anomaly = annual temperature minus the 1981–2010 reference. Positive values indicate warmer years.

In [ ]:
cdo sub cdo/annual_temperature.nc cdo/reference.nc cdo/annual_anomaly.nc

**Inspect the result**

In [ ]:
cdo outputtab,date,value -seltimestep,1/5 cdo/annual_anomaly.nc

## 04 · NCO

NCO starts from the same original inputs. It selects variables and dates,
joins records, computes means and subtracts the reference temperature.

### 04.01 · Select historical months

Select tas for January 1979–December 2014. -v selects the variable; -d selects dates. Coordinates and linked time bounds are retained automatically. The metadata shows 432 monthly records.

In [ ]:
ncks -v tas -d time,1979-01-01,2014-12-31 data/historical.nc nco/historical.nc

**Inspect the result**

In [ ]:
ncks -m -C -v tas nco/historical.nc

### 04.02 · Select SSP5-8.5 months

Select January 2015–December 2024 from the original SSP5-8.5 file. The result contains 120 monthly records on the same grid.

In [ ]:
ncks -v tas -d time,2015-01-01,2024-12-31 data/ssp585.nc nco/ssp585.nc

**Inspect the result**

In [ ]:
ncks -m -C -v tas nco/ssp585.nc

### 04.03 · Join historical and SSP5-8.5

ncrcat joins the monthly records in the listed order. The output contains 552 consecutive months from 1979 through 2024.

In [ ]:
ncrcat nco/historical.nc nco/ssp585.nc nco/merged.nc

**Inspect the result**

In [ ]:
ncks -m -C -v tas nco/merged.nc

### 04.04 · Convert Kelvin to Celsius

ncap2 -s evaluates an arithmetic expression. Subtract 273.15 and label the values degC. double(tas) keeps double precision for subsequent calculations. Inspect the first month at one grid point.

In [ ]:
ncap2 -s 'tas=double(tas)-273.15;tas@units="degC";' nco/merged.nc nco/celsius.nc

**Inspect the result**

In [ ]:
ncks -H -C -v tas -d time,0 -d lat,0 -d lon,0 nco/celsius.nc

### 04.05 · Calculate the monthly Arctic temperature

Select latitude centres from 65°N to 90°N. ncwa -a lat,lon averages both spatial dimensions, giving every selected grid point equal weight. Inspect the first twelve monthly temperatures.

In [ ]:
ncwa -a lat,lon -d lat,65.0,90.0 nco/celsius.nc nco/arctic_monthly.nc

**Inspect the result**

In [ ]:
ncks -H -C -v tas -d time,0,11 nco/arctic_monthly.nc

### 04.06 · Calculate the annual Arctic temperature

ncra --mro produces one mean for each group of twelve monthly records. Every month has equal weight. -C -v tas,time retains the temperature and time coordinate without carrying monthly interval bounds into the annual output. Check that there are 46 annual values.

In [ ]:
ncra --mro -C -v tas,time -d time,,,12,12 nco/arctic_monthly.nc nco/annual_temperature.nc

**Inspect the result**

In [ ]:
ncks -m -C -v tas nco/annual_temperature.nc

### 04.07 · Calculate the 1981–2010 reference temperature

Select the thirty annual values from 1981 through 2010 and average them over time. The result is one Arctic reference temperature. This baseline uses the same complete years and equal monthly weights as the shared calculation.

In [ ]:
ncwa -a time -d time,1981-01-01,2010-12-31 nco/annual_temperature.nc nco/reference.nc

**Inspect the result**

In [ ]:
ncks -H -C -v tas nco/reference.nc

### 04.08 · Subtract the reference temperature

ncdiff subtracts the second file from the first. The single reference temperature is subtracted from every annual value. Positive anomalies indicate years warmer than the 1981–2010 reference. Inspect the first five annual anomalies.

In [ ]:
ncdiff nco/annual_temperature.nc nco/reference.nc nco/annual_anomaly.nc

**Inspect the result**

In [ ]:
ncks -H -C -v tas -d time,0,4 nco/annual_anomaly.nc

### 04.09 · Label the annual anomaly

Label tas as an air-temperature anomaly. Remove links to the monthly bounds, auxiliary coordinates and external area variable that are absent from this annual series. The time coordinate identifies each year; this file does not claim interval bounds. The anomaly values remain unchanged.

In [ ]:
ncatted -a bounds,time,d,, -a coordinates,tas,d,, -a cell_measures,tas,d,, -a standard_name,tas,o,c,air_temperature_anomaly nco/annual_anomaly.nc

**Inspect the result**

In [ ]:
ncks -m -C -v tas nco/annual_anomaly.nc

## 05 · Compare the annual series

**These three files supply the three curves and their climate stripes.**
They contain results already calculated in the preceding steps.
Paths below are relative to the current results folder.

| Plot label | Data file | Variable | What it contains |
|---|---|---|---|
| Python reference | `python/arctic_annual.nc` | `tas_anom` | Independent Python calculation using monthly climatology and monthly anomalies |
| CDO | `cdo/annual_anomaly.nc` | `tas` | Annual anomalies calculated by the CDO commands above |
| NCO | `nco/annual_anomaly.nc` | `tas` | Annual anomalies calculated by the NCO commands above |

All three select 1979–2014 from `data/historical.nc` and 2015–2024 from
`data/ssp585.nc`, using the same ACCESS-CM2 data. Each plotted series contains **46 annual
Arctic anomalies (65–90°N, 1979–2024), in °C relative to 1981–2010**, with
equal grid-point and monthly weights. The Python reference is calculated
independently in this lesson; it is not an ESMValTool output file.

[Open the full comparison and plotting script: compare_short_annual.py](../scripts/compare_short_annual.py)

The Python source below is the actual script used by the next calculation cell.

In [ ]:
from IPython.display import Code
display(Code(filename=str(ROOT / "scripts/compare_short_annual.py"), language="python"))

Run the next cell to list the input files and check that the 46 annual values
agree. The script also reads the CDO and NCO `annual_temperature.nc` and
`reference.nc`, plus `python/climatology.nc`, to check the temperatures and
baseline; these files do not supply additional curves. It checks the near-zero
mean anomaly in 1981–2010 as well. This is an annual-result comparison; the
full notebooks contain the five-product comparison.

In [ ]:
subprocess.run([sys.executable, str(ROOT / "scripts/compare_short_annual.py"), str(RUN)], check=True)
report = json.loads((RUN / "short_validation.json").read_text())
assert report["passed"]
print("Both annual series passed; 46 years per route.")

## 06 · Read the stripes

The same color scale is used for all three routes. Each stripe represents one
annual Arctic temperature anomaly. Positive values are warmer than 1981–2010.
These are results from one model realization, not observations.

In [ ]:
display(Image(filename=str(RUN / "short_comparison.png")))

## Continue with monthly fields

Use the [complete hands-on workflow](04_monthly_anomaly_hands_on.ipynb) for
monthly reference fields, monthly anomalies, a duration-weighted map and
experiments with the Arctic boundary or annual weighting.

Download your edited notebook and results before closing a Binder session.